# Dataset creation pipeline: steps 1 to 4

This notebook runs the dataset creation scripts in order:

1. Convert a ROS bag to IMU and ground-truth CSV files
2. Plot the raw CSV files for quick inspection
3. Clean the IMU and ground-truth CSV files
4. Align cleaned ground truth to the cleaned IMU timeline

Only edit `BAG_PATH` in the setup cell.

In [32]:
# Setup cell: change BAG_PATH to process a different rosbag.
from pathlib import Path
import subprocess
import sys

WORKSPACE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PIPELINE_DIR = WORKSPACE_ROOT / "scripts" / "pipeline"
BAG_PATH = WORKSPACE_ROOT / "data" / "rosbags" / "2026-07-08-calibrated_AI" / "flight_2026-07-01-10-36-01.bag"
OUTPUT_DIR = WORKSPACE_ROOT / "results" / "notebook_runs" / BAG_PATH.stem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Derived filenames used by scripts 1 to 4.
BASE = OUTPUT_DIR / BAG_PATH.stem
IMU_CSV = Path(f"{BASE}_imu.csv")
GT_CSV = Path(f"{BASE}_gt.csv")
IMU_CLEAN_CSV = Path(f"{BASE}_imu_clean.csv")
GT_CLEAN_CSV = Path(f"{BASE}_gt_clean.csv")
IMU_200HZ_CSV = Path(f"{BASE}_imu_clean_200hz.csv")
GT_ALIGNED_CSV = Path(f"{BASE}_gt_clean_aligned.csv")
RAW_PLOTS_DIR = OUTPUT_DIR / "plots_raw"

def run_command(command):
    """Run one pipeline command and stop the notebook if it fails."""
    print("Running:", " ".join(str(part) for part in command))
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    result.check_returncode()

print(f"Rosbag: {BAG_PATH}")
print(f"Raw IMU CSV: {IMU_CSV}")
print(f"Raw GT CSV: {GT_CSV}")
print(f"Clean IMU CSV: {IMU_CLEAN_CSV}")
print(f"Clean GT CSV: {GT_CLEAN_CSV}")
print(f"200 Hz IMU CSV: {IMU_200HZ_CSV}")
print(f"Aligned GT CSV: {GT_ALIGNED_CSV}")

Rosbag: flight_2026-07-01-10-36-01.bag
Raw IMU CSV: flight_2026-07-01-10-36-01_imu.csv
Raw GT CSV: flight_2026-07-01-10-36-01_gt.csv
Clean IMU CSV: flight_2026-07-01-10-36-01_imu_clean.csv
Clean GT CSV: flight_2026-07-01-10-36-01_gt_clean.csv
200 Hz IMU CSV: flight_2026-07-01-10-36-01_imu_clean_200hz.csv
Aligned GT CSV: flight_2026-07-01-10-36-01_gt_clean_aligned.csv


## 1. ROS bag to CSV

This reads the IMU topic plus the AIIMU1 VRPN pose and twist topics from the `.bag` file and saves two CSV files: one for IMU data and one for ground truth.

In [33]:
# Step 1: convert the rosbag into IMU and GT CSV files.
run_command([
    "python3",
    str(PIPELINE_DIR / "1_rosbag_to_csv_vrpn.py"),
    "--bag",
    str(BAG_PATH),
    "--imu_out",
    str(IMU_CSV),
    "--gt_out",
    str(GT_CSV),
])

Running: python3 1_rosbag_to_csv_vrpn.py --bag flight_2026-07-01-10-36-01.bag --imu_out flight_2026-07-01-10-36-01_imu.csv --gt_out flight_2026-07-01-10-36-01_gt.csv
Saved IMU CSV to flight_2026-07-01-10-36-01_imu.csv
IMU rows: 10652
Saved GT CSV to flight_2026-07-01-10-36-01_gt.csv
GT rows: 7147
VRPN pose rows: 7147
VRPN twist rows: 7130



## 3. Clean CSV files

This removes physically unreasonable values, applies a sliding median filter, and fills gaps with linear interpolation.

In [34]:
# Step 3: clean both raw CSV files.
# Set --savgol_window to 0 if you want to disable Savitzky-Golay smoothing.
run_command([
    "python3",
    str(PIPELINE_DIR / "3_csv_dataset_cleaner.py"),
    "--imu_csv",
    str(IMU_CSV),
    "--gt_csv",
    str(GT_CSV),
    "--savgol_window",
    "21",
    "--savgol_polyorder",
    "2",
])

Running: python3 3_csv_dataset_cleaner.py --imu_csv flight_2026-07-01-10-36-01_imu.csv --gt_csv flight_2026-07-01-10-36-01_gt.csv --savgol_window 21 --savgol_polyorder 2
Saved cleaned CSV to flight_2026-07-01-10-36-01_imu_clean.csv
Physical pruning counts:
  gyro_x: 0
  gyro_y: 0
  gyro_z: 0
  acc_x: 0
  acc_y: 0
  acc_z: 2
Median filter counts:
  gyro_x: 4
  gyro_y: 11
  gyro_z: 0
  acc_x: 12
  acc_y: 7
  acc_z: 12
Saved cleaned CSV to flight_2026-07-01-10-36-01_gt_clean.csv
Physical pruning counts:
  vel_x: 0
  vel_y: 0
  vel_z: 0
Median filter counts:
  pos_x: 0
  pos_y: 0
  pos_z: 0
  vel_x: 0
  vel_y: 1
  vel_z: 2



## 2. Plot raw CSV data

This creates quick PNG plots for the raw IMU and ground-truth data so you can visually inspect the signals before cleaning.

In [41]:
# Step 2: plot raw IMU and GT CSV files.
run_command([
    "python3",
    str(PIPELINE_DIR / "2_plot_csv_data.py"),
    "--imu_csv",
    str(IMU_CSV),
    "--gt_csv",
    str(GT_CSV),
    "--out_dir",
    str(RAW_PLOTS_DIR),
])

Running: python3 2_plot_csv_data.py --imu_csv /home/tl/Downloads/Dataset_preparation_Airimu/AI_CSVs_Rishi/flight_2026-07-01-10-28-18/flight_2026-07-01-10-28-18_imu_clean_200hz.csv --gt_csv /home/tl/Downloads/Dataset_preparation_Airimu/AI_CSVs_Rishi/flight_2026-07-01-10-28-18/flight_2026-07-01-10-28-18_gt_clean_aligned.csv --out_dir plots_flight_2026-07-01-10-28-18_raw


## 4. Align ground truth to IMU timeline

This interpolates the cleaned IMU data to 200 Hz, compensates the Vicon latency, and resamples cleaned ground truth onto the 200 Hz IMU timestamps using nearest-neighbor alignment.

In [ ]:
# Step 4: interpolate cleaned IMU to 200 Hz, then align cleaned GT to that timeline.
run_command([
    "python3",
    str(PIPELINE_DIR / "4_align_gt_and_imu.py"),
    "--imu_csv",
    str(IMU_CLEAN_CSV),
    "--gt_csv",
    str(GT_CLEAN_CSV),
    "--resample_imu_hz",
    "200",
    "--max_interp_gap",
    "0.05",
    "--imu_resampled_out",
    str(IMU_200HZ_CSV),
    "--out",
    str(GT_ALIGNED_CSV),
])

Running: python3 4_align_gt_and_imu.py --imu_csv flight_2026-07-01-10-36-01_imu_clean.csv --gt_csv flight_2026-07-01-10-36-01_gt_clean.csv --resample_imu_hz 200 --max_interp_gap 0.05 --imu_resampled_out flight_2026-07-01-10-36-01_imu_clean_200hz.csv --out flight_2026-07-01-10-36-01_gt_clean_aligned.csv
Saved aligned GT CSV to flight_2026-07-01-10-36-01_gt_clean_aligned.csv
Saved resampled IMU CSV to flight_2026-07-01-10-36-01_imu_clean_200hz.csv

Original timelines:
  IMU: rows=10652, duration=70.589s, rate=150.89 Hz, start=1782902163.269966, end=1782902233.858576
  GT:  rows=7147, duration=70.606s, rate=101.21 Hz, start=1782902163.260104, end=1782902233.865605
  IMU interpolated to 200 Hz:
       rows=14118, duration=70.585s, rate=200.00 Hz, start=1782902163.269966, end=1782902233.854966
  GT after latency compensation (0.025000s):
       rows=7147, duration=70.606s, rate=101.21 Hz, start=1782902163.235104, end=1782902233.840605

Aligned output:
  GT aligned to IMU: rows=14118, durati

Done. The main output files should be named like:

- `*_imu.csv` and `*_gt.csv` for the raw extracted files
- `*_imu_clean.csv` and `*_gt_clean.csv` for the cleaned files
- `*_imu_clean_200hz.csv` for the interpolated 200 Hz IMU file
- `*_gt_clean_aligned.csv` for the ground-truth file aligned to the 200 Hz IMU timeline
- `plots_<bag_name>_raw/` for the raw inspection plots